# 📚 Guide Complet & Explication de Tous les Fichiers et Codes du Projet
> **Projet : SalesTeam AI (Système de Recommandation Commerciale B2B pour la filiale LSAT)**  
> **Objectif de ce Notebook** : Parcourir **chaque fichier source du projet**, expliquer son **idée directrice**, son **fonctionnement technique détaillé** (fonctions, algorithmes, paramètres) et **ses relations / dépendances** avec les autres composants.

---

## 🗺️ Cartographie Globale des Dépendances du Projet

Le schéma ci-dessous illustre comment les 17 fichiers de code Python et React interagissent entre eux :

<div align="center" style="background-color:#161b22; padding:15px; border-radius:10px; border:1px solid #30363d; margin: 15px 0;">
  <img src="architecture_diagram.png" alt="Cartographie Architecture SalesTeam AI" style="max-width:100%; height:auto; border-radius:8px;" />
</div>

> 💡 *Fichier interactif disponible également dans votre navigateur : ouvrez **`notebooks/architecture_diagram.html`**.*

<details>
<summary><b>🔍 Cliquez ici pour voir le code source du schéma (Mermaid)</b></summary>

```mermaid
flowchart TD
    subgraph S1 [1. Données Brutes & Nettoyage - src/data]
        L["loader.py"] -->|Tables brutes fusionnées| CL["cleaner.py"]
    end

    subgraph S2 [2. Feature Engineering & Cible - src/features & src/models]
        CL -->|commandes_clean, lignes_clean| FE["feature_engineering.py"]
        CL -->|commandes_clean, lignes_clean| TB["target_builder.py"]
        FE -->|cat_quarterly_index.json| TB
    end

    subgraph S3 [3. Entraînement des Modèles ML - src/models]
        TB -->|training_set.csv| TC["train_classifier.py"]
        TB -->|training_set.csv achats réels| TR["train_regressor.py"]
        TR -.->|Étude seuil CV| CC["compare_cv_threshold.py"]
    end

    subgraph S4 [4. Services Métier & Inférence - src/services]
        TC -->|classifier_lsat.joblib| REC["recommendation.py"]
        TR -->|regressor_lsat.joblib| REC
        FE -->|cat_quarterly_index.json| REC
        REC -->|Demande explication| EXP["explanation.py"]
        REC -->|Extraction contexte| DC["deep_context.py"]
        DC -->|Historique & Rangs| EXP
        FB_SERV["feedback.py"] -->|Enregistrement retours| RETRAIN[("Données ré-entraînement")]
    end

    subgraph S5 [5. API REST FastAPI - src/api]
        SCH["schemas.py (Pydantic)"] --> R_REC["routes/recommend.py"]
        SCH --> R_CLI["routes/clients.py"]
        SCH --> R_FB["routes/feedback.py"]
        SCH --> R_ADM["routes/admin.py"]
        
        R_REC --> REC
        R_CLI --> REC
        R_FB --> FB_SERV
        R_ADM --> REC
        
        MAIN["main.py (FastAPI App)"] --> R_REC
        MAIN --> R_CLI
        MAIN --> R_FB
        MAIN --> R_ADM
    end

    subgraph S6 [6. Frontend Web Commercial - frontend/src]
        MAIN -.->|API REST HTTP/JSON| APP["App.jsx (React)"]
        APP --> MAIN_JSX["main.jsx"]
        APP --> CSS["App.css / index.css"]
    end
```

</details>


In [1]:
# Vérification automatique de l'existence et du volume de code de tous les fichiers du projet
import os
import pandas as pd

project_files = [
    # Data
    ("src/data/loader.py", "Couche Données", "Ingestion et fusion brute multi-fichiers"),
    ("src/data/cleaner.py", "Couche Données", "Nettoyage, filtrage LSAT et déduplication"),
    # Features
    ("src/features/feature_engineering.py", "Features", "Calcul des métriques RFM, délais et saisonnalité"),
    # Models
    ("src/models/target_builder.py", "Modélisation", "Construction du dataset visit-level anti-fuite temporelle"),
    ("src/models/train_classifier.py", "Modélisation", "Entraînement du classifieur XGBoost d'intention d'achat"),
    ("src/models/train_regressor.py", "Modélisation", "Entraînement du régresseur XGBoost de volume (MAE Loss)"),
    ("src/models/compare_cv_threshold.py", "Modélisation", "Benchmark et validation empirique du seuil de variabilité CV"),
    # Services
    ("src/services/recommendation.py", "Services", "Moteur d'inférence en temps réel, reclassement et clamping"),
    ("src/services/explanation.py", "Services", "Génération des explications LLM (Groq) et fallback Python"),
    ("src/services/deep_context.py", "Services", "Extraction du contexte profond (historique, décomposition score)"),
    ("src/services/feedback.py", "Services", "Enregistrement des retours d'acceptation/rejet commerciaux"),
    # API
    ("src/api/main.py", "API REST", "Point d'entrée de l'application FastAPI"),
    ("src/api/schemas.py", "API REST", "Contrats de données Pydantic (validation entrées/sorties)"),
    ("src/api/routes/recommend.py", "API REST", "Routes HTTP de recommandation et d'explication détaillée"),
    ("src/api/routes/clients.py", "API REST", "Route HTTP listant les 569 clients éligibles"),
    ("src/api/routes/feedback.py", "API REST", "Route HTTP de soumission des retours commerciaux"),
    ("src/api/routes/admin.py", "API REST", "Routes HTTP de monitoring et gestion du cache"),
    # Frontend
    ("frontend/src/App.jsx", "Frontend Web", "Composant principal React : cockpit commercial interactif"),
    ("frontend/src/main.jsx", "Frontend Web", "Point de montage React DOM"),
    ("frontend/vite.config.js", "Frontend Web", "Configuration du bundler Vite et proxy API")
]

file_stats = []
for path, module, desc in project_files:
    exists = os.path.exists(path) or os.path.exists(os.path.join('..', path))
    resolved = path if os.path.exists(path) else os.path.join('..', path)
    if exists:
        with open(resolved, 'r', encoding='utf-8') as f:
            lines = len(f.readlines())
    else:
        lines = 0
    file_stats.append({
        "Fichier": path,
        "Module": module,
        "Rôle Résumé": desc,
        "Lignes de Code": lines,
        "Présent": "✅" if exists else "❌"
    })

df_files = pd.DataFrame(file_stats)
df_files


,Fichier,Module,Rôle Résumé,Lignes de Code,Présent
0,src/data/loader.py,Couche Données,Ingestion et fusion brute multi-fichiers,143,✅
1,src/data/cleaner.py,Couche Données,"Nettoyage, filtrage LSAT et déduplication",140,✅
2,src/features/feature_engineering.py,Features,"Calcul des métriques RFM, délais et saisonnalité",244,✅
3,src/models/target_builder.py,Modélisation,Construction du dataset visit-level anti-fuite...,287,✅
4,src/models/train_classifier.py,Modélisation,Entraînement du classifieur XGBoost d'intentio...,232,✅
5,src/models/train_regressor.py,Modélisation,Entraînement du régresseur XGBoost de volume (...,476,✅
6,src/models/compare_cv_threshold.py,Modélisation,Benchmark et validation empirique du seuil de ...,214,✅
7,src/services/recommendation.py,Services,"Moteur d'inférence en temps réel, reclassement...",586,✅
8,src/services/explanation.py,Services,Génération des explications LLM (Groq) et fall...,644,✅
9,src/services/deep_context.py,Services,"Extraction du contexte profond (historique, dé...",482,✅


---
# 📦 Module 1 — Couche d'Ingestion & Nettoyage des Données (`src/data/`)

## 1.1 `src/data/loader.py` — Ingestion et Fusion Multi-Sources

### 💡 Idée et Raison d'Être
Dans un environnement ERP Navision, les données commerciales ne sont jamais livrées dans une table unique. Elles sont découpées en en-têtes de factures, lignes d'articles et fiches clients.
`loader.py` a pour responsabilité unique de **lire ces fichiers hétérogènes Excel/CSV**, de **valider leur format** et de **réaliser la jointure composite** pour former la grande table analytique brute (`main_table.csv`).

### ⚙️ Fonctionnement Technique Détaillé
Le fichier contient 4 fonctions principales :
1. `load_factures(filepath)` : Charge `commande_phonesTech_lsat.xlsx`. Convertit les dates en `pd.to_datetime`, nettoie les colonnes d'en-tête (`code_facture`, `code_client`, `date_facture`, `societe`).
2. `load_lignes(filepath)` : Charge `commande_lines_lsat.xlsx`. Lit les 78 530 lignes de commandes avec les volumes (`quantite`), les codes articles (`code_article`), les libellés et les familles de produits.
3. `load_clients(filepath)` : Charge `client_lat_lng_lsat.xlsx`. Récupère les coordonnées GPS (`latitude`, `longitude`, `ville`, `gouvernorat`).
4. `build_main_table(...)` : Réalise la jointure cruciale :
   - **Jointure critique sur clé composite** : `pd.merge(factures, lignes, on=['code_facture', 'societe'], how='inner')`.
   - *Pourquoi composite ?* Dans Navision, un même numéro de facture `FA2024-001` peut exister chez `LSAT` et chez `PHONESTECH`. Sans la clé `societe`, la jointure produit un produit cartésien désastreux.

### 🔗 Relations & Dépendances
* **Appelé par** : Les scripts de préparation de données ou directement par `cleaner.py`.
* **Fichiers qu'il lit** : `data/raw/commande_phonesTech_lsat.xlsx`, `data/raw/commande_lines_lsat.xlsx`, `data/raw/client_lat_lng_lsat.xlsx`.
* **Fichiers qu'il produit** : `data/processed/main_table.csv`.

---

## 1.2 `src/data/cleaner.py` — Nettoyage, Normalisation & Filtrage Métier

### 💡 Idée et Raison d'Être
Les données brutes d'ERP contiennent du bruit qui fausse l'apprentissage automatique : avoirs comptables négatifs, doublons de saisie, espaces invisibles dans les codes et clients sporadiques n'ayant commandé qu'une seule fois.
`cleaner.py` applique toutes les règles de **purification métier** pour transformer les données brutes en jeux de données prêts pour l'ingénierie des caractéristiques.

### ⚙️ Fonctionnement Technique Détaillé
1. `clean_commandes(df)` :
   - Filtre strictement la filiale cible : `df = df[df['societe'] == 'LSAT']`.
   - Normalise les identifiants : `code_client.str.strip().str.upper()`.
   - Extrait les dimensions calendaires : `mois`, `annee`, `jour_semaine`, `trimestre`.
   - **Filtre des clients actifs (`MIN_HISTORY_ORDERS = 3`)** : Écarte les acheteurs ponctuels qui n'ont commandé qu'une ou deux fois en 2 ans et demi (passage de 737 clients bruts à **569 clients récurrents qualifiés**).
2. `clean_lignes(df)` :
   - Élimine le bruit comptable : Supprime les lignes où `quantite <= 0` ou `quantite.isna()`.
   - **Agrégation multi-lignes** : Si un article est mentionné sur deux lignes d'une même facture (conditionnements différents), leurs quantités sont sommées : `groupby(['code_facture', 'societe', 'code_article'])['quantite'].sum()`.
   - Imputation catégorielle : Les articles orphelins sans sous-famille reçoivent la valeur `'AUTRES'`.
3. `clean_gps(df)` : Valide les plages de coordonnées géographiques pour la Tunisie (latitude $\in [30, 38]$, longitude $\in [7, 12]$).
4. `clean_all(...)` : Orchestre l'ensemble du pipeline et sauvegarde les fichiers propres.

### 🔗 Relations & Dépendances
* **Appelle** : `src/data/loader.py` pour ingérer les tables.
* **Appelé par** : `src/features/feature_engineering.py` et `src/models/target_builder.py`.
* **Fichiers produits** : `data/processed/commandes_clean.csv`, `data/processed/lignes_clean.csv`, `data/processed/gps_clean.csv`.


---
# 🔬 Module 2 — Ingénierie des Caractéristiques (`src/features/`)

## 2.1 `src/features/feature_engineering.py` — Calculateur de Signaux Comportementaux

### 💡 Idée et Raison d'Être
Un algorithme de Machine Learning ne comprend pas directement des listes de factures. Il a besoin d'indicateurs quantitatifs qui synthétisent le comportement d'achat de chaque client pour chaque produit.
`feature_engineering.py` transforme les séries chronologiques de commandes en une **matrice de caractéristiques comportementales** (RFM, régularité, tendance, saisonnalité).

### ⚙️ Fonctionnement Technique Détaillé
Le fichier comprend 4 composantes majeures :

1. **Calcul de la Saisonnalité Catégorielle (`build_categorical_quarterly_index`)** :
   - Agrège les volumes vendus par famille de produit et par trimestre civil ($Q_1$ à $Q_4$) sur les années 2024 à 2026.
   - Formule du coefficient :
     $$S_{\text{famille}, Q} = \frac{\bar{Q}_{\text{famille}, Q}}{\bar{Q}_{\text{famille}, \text{annuel}}}$$
   - Sauvegarde le dictionnaire dans `cat_quarterly_index.json` (ex: Accessoires en été $Q_3 = 1.18$, Smartphones en fin d'année $Q_4 = 1.12$).

2. **Calcul de la Matrice Principale (`build_feature_matrix`)** :
   - Pour chaque couple `(code_client, code_article)`, calcule :
     - `frequency` : Nombre total de commandes historiques.
     - `total_qty` : Volume cumulé vie client.
     - `avg_qty` : Volume moyen par commande.
     - `median_qty` : Volume médian (robuste aux pics).
     - `std_qty` : Écart-type des quantités (volatilité).
     - `min_qty` & `max_qty` : Bornes historiques de commande.
     - `last_qty` : Quantité commandée la dernière fois.
     - `recency_days` : Nombre de jours depuis la dernière commande.
     - `avg_delay_days` : Intervalle moyen entre commandes successives.
     - `recency_relative` : Ratio $\frac{\text{recency\_days}}{\text{avg\_delay\_days}}$.
     - `trend` : Pente de régression des 3 dernières commandes $\frac{Q_{\text{last}} - Q_{\text{first}}}{Q_{\text{mean}}}$.

3. **Nettoyage des Variables Mortes** :
   - Suite à notre audit d'optimisation, les variables redondantes (`company_encoded`, `days_since_first_order`, `nb_clients`, `is_bulk_product`) ont été épurées pour concentrer le modèle sur les variables à fort pouvoir prédictif.

### 🔗 Relations & Dépendances
* **Lit** : `commandes_clean.csv`, `lignes_clean.csv`.
* **Produit** : `data/processed/feature_matrix.csv`, `data/processed/cat_quarterly_index.json`.
* **Consommé par** : `src/models/target_builder.py` et en temps réel par `src/services/recommendation.py`.


---
# 🧠 Module 3 — Modélisation Prédictive & Entraînement (`src/models/`)

## 3.1 `src/models/target_builder.py` — Construction du Dataset Visit-Level Anti-Fuite

### 💡 Idée et Raison d'Être
C'est le fichier **le plus stratégique pour la rigueur scientifique du projet**.
Si l'on calcule les features d'un client à l'instant $T$ en utilisant les données de toute l'année, le modèle « triche » en connaissant le futur (*data leakage*).
`target_builder.py` construit une matrice d'apprentissage **visite par visite** en respectant rigoureusement la flèche du temps.

### ⚙️ Fonctionnement Technique Détaillé
1. `build_visit_level_dataset(...)` :
   - Parcourt chaque visite réelle $t$ d'un client (chaque facture passée).
   - **Coupure temporelle stricte (Point-in-Time Cutoff)** : Pour calculer les features associées à la visite $t$, la fonction ne sélectionne **QUE** les transactions strictement antérieures : $\mathcal{H}_{<t} = \{ t' < t \}$.
2. **Génération des Exemples Positifs & Négatifs** :
   - **Positifs ($Y=1$, `target_bought=1`)** : Tout article effectivement présent dans le bon de commande de la visite $t$. La variable `target_qty` prend la quantité réelle commandée.
   - **Négatifs ($Y=0$, `target_bought=0`)** : Tout article que le client a l'habitude de commander mais qu'il **n'a pas acheté lors de cette visite précise**. La variable `target_qty` vaut $0$.
3. **Volume Résultant** :
   - 1 222 876 lignes.
   - 37 424 exemples positifs (**3.06%**).
   - 1 185 452 exemples négatifs (**96.94%**).

### 🔗 Relations & Dépendances
* **Lit** : `commandes_clean.csv`, `lignes_clean.csv`, `cat_quarterly_index.json`.
* **Produit** : `data/processed/training_set.csv`.
* **Alimente** : `train_classifier.py` et `train_regressor.py`.

---

## 3.2 `src/models/train_classifier.py` — Entraînement du Modèle 1 (Intention d'Achat)

### 💡 Idée et Raison d'Être
Répond à la question : *« Ce client va-t-il commander ce produit lors de sa visite aujourd'hui ? »*
Il entraîne un classifieur binaire XGBoost hautement optimisé pour gérer l'extrême rareté des achats (3% de positifs).

### ⚙️ Fonctionnement Technique Détaillé
1. **Sélection des 9 Features Clés** :
   `['recency_days', 'recency_relative', 'frequency', 'avg_delay_days', 'total_qty', 'avg_qty', 'std_qty', 'min_qty', 'cat_quarterly_coef']`.
2. **Découpage Temporel Hors-du-Temps (Out-of-Time Split)** :
   - Train : Janvier 2024 à Février 2026 (544 clients).
   - Test : Mars 2026 à Juin 2026 (288 clients sur commandes futures).
3. **Gestion du Déséquilibre Extrême** :
   - Utilise `scale_pos_weight = 29.36` (calibré exactement sur $\frac{N_{\text{négatifs}}}{N_{\text{positifs}}}$). Cela force le modèle à accorder autant d'importance aux 3% d'achats qu'aux 97% de non-achats.
4. **Hyperparamètres & Métriques** :
   - `max_depth=5`, `learning_rate=0.05`, `n_estimators=300` (meilleure itération = 19).
   - **ROC-AUC obtenu : 0.8668 (86.7%)**.
   - **PR-AUC obtenu : 0.0902** (3x supérieur au hasard).
5. **Sauvegarde** : Enregistre le binaire `classifier_lsat.joblib` et son fichier de traçabilité `classifier_lsat_metadata.json`.

### 🔗 Relations & Dépendances
* **Lit** : `data/processed/training_set.csv`.
* **Produit** : `src/models/classifier_lsat.joblib` et `classifier_lsat_metadata.json`.
* **Consommé par** : `src/services/recommendation.py`.

---

## 3.3 `src/models/train_regressor.py` — Entraînement du Modèle 2 (Volume Suggéré)

### 💡 Idée et Raison d'Être
Répond à la question : *« Si le client achète, combien d'unités suggérer ? »*
Il entraîne un régresseur XGBoost avec fonction de perte absolue (MAE), sécurisé par un ensemble de garde-fous statistiques.

### ⚙️ Fonctionnement Technique Détaillé
1. **Population d'Entraînement Exclusive** :
   - N'est entraîné **que sur les achats réels** (`target_qty > 0`, soit 37 424 lignes). Cela résout définitivement le biais de la régression zéro-gonflée.
2. **Fonction de Perte MAE (`reg:absoluteerror`)** :
   - Optimise la médiane conditionnelle plutôt que la moyenne quadratique (MSE). Ainsi, une commande exceptionnelle de 500 pièces ne fait pas dériver les suggestions courantes.
3. **Sélection des 10 Features** :
   `['avg_qty', 'median_qty', 'last_qty', 'max_qty', 'std_qty', 'min_qty', 'frequency', 'recency_days', 'avg_delay_days', 'cat_quarterly_coef']`.
4. **Résultats vs Baseline Historique** :
   - **MAE IA : 6.53 unités** vs **7.14 unités** pour la moyenne historique (**+8.56% d'amélioration**).
   - **RMSE IA : 28.16 unités** vs **30.87 unités** (**+8.77% d'amélioration**).
5. **Garde-Fous Intégrés** :
   - Clamping : $\hat{Q} \in [\max(1, 0.5 \times \text{min}), 2.0 \times \text{max}]$.
   - Bascule de volatilité : si $CV = \frac{\text{std}}{\text{avg}} > 1.0 \rightarrow \lceil \text{avg\_qty} \rceil$.

### 🔗 Relations & Dépendances
* **Lit** : `data/processed/training_set.csv`.
* **Produit** : `src/models/regressor_lsat.joblib` et `regressor_lsat_metadata.json`.
* **Consommé par** : `src/services/recommendation.py`.

---

## 3.4 `src/models/compare_cv_threshold.py` — Étude Expérimentale de Variabilité

### 💡 Idée et Raison d'Être
Ce script sert de banc d'essai expérimental pour déterminer à quel niveau de variabilité ($CV = \text{std}/\text{avg}$) le modèle XGBoost devient moins fiable que la simple moyenne historique.

### ⚙️ Fonctionnement Technique Détaillé
- Teste différents seuils de bascule : $CV \in [0.5, 0.8, 1.0, 1.2, 1.5, \infty]$.
- Compare l'erreur MAE globale pour chaque seuil.
- Démontre mathématiquement que le seuil optimal de bascule est **$CV = 1.0$**, valeur retenue dans notre moteur d'inférence de production.


---
# ⚙️ Module 4 — Couche Métier & Inférence en Temps Réel (`src/services/`)

## 4.1 `src/services/recommendation.py` — Le Moteur Central de Recommandation

### 💡 Idée et Raison d'Être
C'est le **cœur opérationnel de tout le projet**. C'est lui qui reçoit une demande commercial (`client_id`, `date`, `top_n`) et orchestre l'ensemble de la chaîne : génération de candidats, prédictions des deux modèles IA, reclassement par règles métier, calcul des fourchettes et demande d'explications au LLM.

### ⚙️ Fonctionnement Technique Détaillé
1. `_get_artifacts()` : Charge les modèles `.joblib` et les métadonnées une seule fois en mémoire cache (Singleton Pattern).
2. `get_available_clients()` : Renvoie la liste des 569 clients avec leur nombre de commandes, CA total et date de dernière visite.
3. `recommend(request)` :
   - **Étape 1 : Candidats** : Extrait tous les produits déjà commandés par ce client dans l'historique.
   - **Étape 2 : Features Point-in-Time** : Calcule les 12 features pour chaque produit candidat à la date demandée.
   - **Étape 3 : Modèle 1** : Prédit la probabilité brute d'achat $P(Y=1)$.
   - **Étape 4 : Reclassement Multiplicatif** :
     $$\text{Score Final} = P(Y=1) \times \text{Timing Boost} \times \text{Trend Boost}$$
     - *Timing Boost* : $3.0\times$ si retard $\ge 1.5$ cycle, $2.0\times$ si retard $\ge 1.0$, $1.5\times$ si retard $\ge 0.85$.
     - *Trend Boost* : $1.2\times$ si hausse $> +10\%$, $0.7\times$ si baisse $< -20\%$.
   - **Étape 5 : Tri & Sélection Top-K** : Classe par score décroissant et prend les $K$ meilleurs.
   - **Étape 6 : Modèle 2** : Prédit la quantité $\hat{Q}$ pour les articles retenus, applique le seuil $CV > 1.0$ et le clamping.
   - **Étape 7 : Fourchette Commerciale** : Calcule $[\max(1, 0.75 \times \hat{Q}), 1.25 \times \hat{Q}]$.
   - **Étape 8 : Explication** : Déclenche l'appel à `explanation.py`.

### 🔗 Relations & Dépendances
* **Appelle** : `train_classifier.py` (artefacts), `train_regressor.py` (artefacts), `explanation.py`, `deep_context.py`.
* **Appelé par** : `src/api/routes/recommend.py`, `src/api/routes/clients.py`.

---

## 4.2 `src/services/explanation.py` — Générateur d'Arguments Commerciaux (Groq LLM)

### 💡 Idée et Raison d'Être
Un commercial ne fera jamais confiance à un algorithme qui lui dit juste *"Article 305 : 37 pièces"* sans expliquer pourquoi.
`explanation.py` transforme les signaux chiffrés complexes en un **argumentaire de vente persuasif, fluide et naturel**, rédigé en français.

### ⚙️ Fonctionnement Technique Détaillé
1. **Architecture Hybride à Double Régime** :
   - **Régime Primaire (IA Cloud)** : Appelle l'API Groq (`https://api.groq.com/openai/v1/chat/completions`) avec le modèle `llama-3.3-70b-versatile` ou `qwen/qwen3.8-27b`.
     - *Latence* : Ultra-rapide (**0.96s à 1.28s** sur puces LPU).
     - *Structure de réponse* : Analyse en 3 volets :
       1. *Pourquoi ce produit ?* (Rythme, retard de cycle, saisonnalité).
       2. *Pourquoi cette quantité ?* (Historique réel, tendance haussière, fourchette).
       3. *Pourquoi ce classement ?* (Comparaison avec les autres produits).
   - **Régime Secondaire (Fallback Déterministe Python)** :
     - Si l'API est indisponible, sans connexion ou que la clé est absente, `_rule_based_detailed_explanation()` génère instantanément (< 1 ms) un texte structuré parfait basé sur des gabarits dynamiques.
2. **Système de Cache Mémoire LRU** :
   - Calcule un hash MD5 du contexte produit-client.
   - Si la même recommandation est demandée deux fois, la réponse est servie en 0 ms sans consommer de quota API.

### 🔗 Relations & Dépendances
* **Appelé par** : `src/services/recommendation.py` et la route `/api/recommend/detailed-explanation`.
* **Appelle** : Groq Cloud API via `urllib.request`.

---

## 4.3 `src/services/deep_context.py` — Extraction du Contexte Analytique Profond

### 💡 Idée et Raison d'Être
Le LLM a besoin de faits concrets pour ne pas halluciner (dates exactes des dernières commandes, volumes précis, comparaison avec les produits classés 1er ou 3ème).
`deep_context.py` est le **fournisseur de preuves** : il extrait de l'historique tous les faits chiffrés vérifiables.

### ⚙️ Fonctionnement Technique Détaillé
- `_get_order_history(client, article)` : Récupère les 3 dernières commandes avec dates réelles et quantités.
- `_get_score_decomposition(...)` : Isole la part de probabilité pure ($P$), du bonus de retard (`timing_boost`) et du bonus de tendance (`trend_boost`).
- `_get_rank_context(...)` : Récupère les scores des voisins immédiats ($K-1$ et $K+1$) pour expliquer pourquoi ce produit est classé à cette position.

### 🔗 Relations & Dépendances
* **Consommé par** : `src/services/explanation.py` et `src/services/recommendation.py`.
* **Lit** : `commandes_clean.csv`, `lignes_clean.csv`, `training_set.csv`.

---

## 4.4 `src/services/feedback.py` — Collecteur de Retours Terrain

### 💡 Idée et Raison d'Être
Permet d'enregistrer chaque action du commercial sur le terrain (produit validé, quantité modifiée, produit refusé avec raison).
Ces retours alimenteront la boucle MLOps pour ré-entraîner les modèles en continu.

### ⚙️ Fonctionnement Technique Détaillé
- `save_feedback(request)` : Enregistre les événements au format JSON horodaté dans `data/feedback/`.
- `load_feedback_for_retraining()` : Charge l'historique des feedbacks pour pénaliser les produits refusés ou ajuster les volumes cibles.

### 🔗 Relations & Dépendances
* **Appelé par** : `src/api/routes/feedback.py`.


---
# 🌐 Module 5 — Couche API REST FastAPI (`src/api/`)

## 5.1 `src/api/main.py` — Point d'Entrée & Configuration du Serveur

### 💡 Idée et Raison d'Être
Instancie l'application FastAPI, configure les autorisations de sécurité réseau (CORS) pour que le frontend React puisse communiquer avec le backend, et monte l'ensemble des routeurs.

### ⚙️ Fonctionnement Technique Détaillé
- Configure le `CORSMiddleware` (`allow_origins=["*"]`, `allow_methods=["*"]`).
- Monte les sous-routeurs : `/api/recommend`, `/api/clients`, `/api/feedback`, `/api/admin`.
- Endpoint de santé : `GET /health` renvoyant le statut opérationnel et l'état des modèles chargés.

---

## 5.2 `src/api/schemas.py` — Contrats de Données & Typage Strict (Pydantic)

### 💡 Idée et Raison d'Être
Définit la structure exacte de chaque requête et réponse JSON. Si le frontend envoie un format incorrect, Pydantic le rejette automatiquement avec un message d'erreur clair.

### ⚙️ Classes Clés
- `RecommendRequest` : `client_id` (str), `visit_date` (str optionnel), `top_n` (int = 5), `model_type` (str).
- `ProductSuggestion` : `code_article`, `designation`, `categorie`, `score_final`, `quantite_suggeree`, `fourchette_min/max`, `explication_courte`.
- `RecommendResponse` : `client_id`, `date`, `total_suggestions`, `suggestions: list[ProductSuggestion]`.
- `DetailedExplanationRequest` & `DetailedExplanationResponse` : Échange du texte argumenté en 3 parties généré par le LLM.
- `FeedbackItem` & `FeedbackRequest` : `client_id`, `code_article`, `action` (`accepted`/`modified`/`rejected`), `quantite_finale`, `raison`.

---

## 5.3 Les Routeurs (`src/api/routes/`)
* **`recommend.py`** :
  - `POST /api/recommend` : Déclenche l'inférence temps réel et renvoie les Top-K recommandations.
  - `POST /api/recommend/detailed-explanation` : Fournit l'explication détaillée rédigée par le LLM à l'ouverture de la modale.
* **`clients.py`** :
  - `GET /api/clients` : Renvoie la liste complète des 569 clients avec compteurs de commandes et CA.
* **`feedback.py`** :
  - `POST /api/feedback` : Reçoit les validations et refus des commerciaux.
* **`admin.py`** :
  - `GET /api/admin/models/info` : Fournit les métriques et métadonnées d'entraînement des modèles.
  - `POST /api/admin/cache/clear` : Vide le cache mémoire des explications LLM.


---
# 💻 Module 6 — Interface Utilisateur Web Commerciale (`frontend/`)

## 6.1 `frontend/src/App.jsx` — Le Cockpit Commercial React

### 💡 Idée et Raison d'Être
C'est l'interface tactile manipulée par le commercial lors de ses visites.
Elle doit être intuitive, extrêmement rapide, claire et permettre d'agir en moins de 30 secondes devant le client.

### ⚙️ Fonctionnalités & Logique Interne
1. **Recherche & Sélection Client** :
   - Barre de recherche instantanée (code, nom, ville).
   - Boutons de présélection rapide pour les clients fréquents (`CLT070730`, etc.).
2. **Affichage des Cartes de Recommandation** :
   - Badge d'urgence : Pastille rouge *"🔴 Rupture Imminente"* si retard critique de cycle.
   - Badge de tendance : Pastille verte *"📈 En Hausse"* si demande croissante.
   - Ajusteur de quantité tactile : Boutons `[ - ]` et `[ + ]` pour modifier la suggestion IA.
   - Fourchette de négociation visible : Ex: *« Idéal 37 (entre 28 et 46) »*.
3. **Modale d'Explication Détaillée (Générée par LLM)** :
   - Bouton *"✨ Justifier"* : Ouvre une fenêtre popup en 3 volets visuels :
     1. *Pourquoi ce produit ?*
     2. *Pourquoi cette quantité ?*
     3. *Pourquoi ce classement ?*
4. **Validation Commerciale & Feedback** :
   - Bouton vert *"Accepter"* : Valide la ligne dans le panier.
   - Bouton rouge *"Refuser"* : Ouvre un dialogue demandant le motif (*Stock plein*, *Trop cher*, *Produit abandonné*).

---

## 6.2 `frontend/src/main.jsx` & `frontend/vite.config.js`
- `main.jsx` : Monte l'application React dans le nœud HTML DOM `#root`.
- `vite.config.js` : Configure le serveur de développement rapide Vite sur le port `5173` et le reverse proxy vers `http://127.0.0.1:8000`.

---

## 6.3 `frontend/src/App.css` & `index.css`
- Système de design moderne en Vanilla CSS / Utility classes.
- Palette de couleurs professionnelle : Bleu Corporate (`#2563eb`), Vert Succès (`#10b981`), Rouge Alerte (`#ef4444`).
- Effets de cartes en glassmorphism et micro-animations fluides au survol.


---
# 📊 Matrice Récapitulative des Flux Entre Fichiers

| Fichier Source | Appelé par (Consommateurs) | Fichiers Appelés (Dépendances) | Données d'Entrée | Données de Sortie |
| :--- | :--- | :--- | :--- | :--- |
| **`loader.py`** | `cleaner.py` | Fichiers Excel bruts | `commande_*.xlsx` | `main_table.csv` |
| **`cleaner.py`** | `feature_engineering.py`, `target_builder.py` | `loader.py` | `main_table.csv` | `commandes_clean.csv`, `lignes_clean.csv` |
| **`feature_engineering.py`** | `target_builder.py`, `recommendation.py` | `cleaner.py` | `commandes_clean.csv` | `feature_matrix.csv`, `cat_quarterly_index.json` |
| **`target_builder.py`** | `train_classifier.py`, `train_regressor.py` | `cleaner.py`, `cat_quarterly_index.json` | Données nettoyées | `training_set.csv` (1.22M lignes) |
| **`train_classifier.py`** | Pipeline MLOps | `target_builder.py` | `training_set.csv` | `classifier_lsat.joblib`, `classifier_metadata.json` |
| **`train_regressor.py`** | Pipeline MLOps | `target_builder.py` | `training_set.csv` ($Q>0$) | `regressor_lsat.joblib`, `regressor_metadata.json` |
| **`recommendation.py`** | `routes/recommend.py`, `routes/clients.py` | `classifier.joblib`, `regressor.joblib`, `explanation.py`, `deep_context.py` | `RecommendRequest` | `RecommendResponse` (Top-K) |
| **`explanation.py`** | `recommendation.py`, `routes/recommend.py` | Groq Cloud API | Contexte produit/client | Argumentaire en 3 volets (texte) |
| **`deep_context.py`** | `recommendation.py`, `explanation.py` | `commandes_clean.csv`, `lignes_clean.csv` | IDs client & article | Dictionnaire de preuves historiques |
| **`feedback.py`** | `routes/feedback.py` | Système de fichiers | `FeedbackRequest` | `data/feedback/*.json` |
| **`main.py`** | Serveur Uvicorn | Tous les routeurs API | Requêtes HTTP | Réponses HTTP JSON |
| **`App.jsx`** | Navigateur Commercial | API REST FastAPI | Actions utilisateur | Interface graphique interactive |
